<a href="https://colab.research.google.com/github/ArkanUbaidillah/BigData26_B_2411537001_ArkanUbaidillahWarman/blob/main/Praktikum5/BD_P05_Tugas_2411537001_Arkan_Ubaidillah_Warman.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tugas Mandiri Praktikum 5: Inverted Index dengan MapReduce

**Mata Kuliah:** Praktikum Big Data (DIF61142)
**Nama:** Arkan Ubaidillah Warman
**NIM:** 2411537001
**Program Studi:** Informatika, Fakultas Teknologi Informasi, Universitas Andalas



## 1. Menyiapkan lapisan terkurasi



In [1]:
from google.colab import drive
drive.mount("/content/drive")          # ikuti dialog izin yang muncul

import os
DIR_KERJA  = "/content/data"                               # sementara, cepat
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Tugas_Praktikum5"   # permanen
os.makedirs(DIR_KERJA, exist_ok=True)
os.makedirs(DIR_SIMPAN, exist_ok=True)
os.chdir(DIR_KERJA)
print("Folder kerja :", os.getcwd())
print("isi drive :", os.listdir(DIR_SIMPAN))

Mounted at /content/drive
Folder kerja : /content/data
isi drive : []


In [2]:
!pip install -q duckdb pyarrow pandas

In [3]:
import os, re, json, zlib, collections
from pathlib import Path
import urllib.request
import pandas as pd

URL_DATA = ("https://raw.githubusercontent.com/ibamibrahim/"
            "dataset-judul-berita-indonesia/master/detik_news_title.csv")
if not os.path.exists("detik_news_title.csv"):
    urllib.request.urlretrieve(URL_DATA, "detik_news_title.csv")

mentah = pd.read_csv("detik_news_title.csv").rename(
    columns={"title": "judul", "category": "kategori", "date": "tanggal"})
mentah["tanggal"] = pd.to_datetime(mentah["tanggal"], format="%m/%d/%Y")

bersih = mentah.copy()
bersih["judul"] = bersih["judul"].str.replace(r"\s+", " ", regex=True).str.strip()
bersih = bersih.drop_duplicates(subset="judul", keep="first")
bersih = bersih[bersih["judul"].str.split().str.len().between(3, 30)].copy()
bersih["bulan"] = bersih["tanggal"].dt.strftime("%Y-%m")
bersih["id"] = bersih["url"].str.extract(r"/d-(\d+)")[0].astype("Int64")
bersih["url"] = bersih["url"].where(bersih["url"].str.contains(r"/d-\d+", regex=True))

TERKURASI = "lapisan_terkurasi"
bersih.to_parquet(TERKURASI, partition_cols=["kategori"], index=False)
splits = sorted(Path(TERKURASI).rglob("*.parquet"))
print("Baris bersih:", f"{len(bersih):,}", "| input split:", len(splits))
print("Judul tanpa id per kategori:", bersih.loc[bersih["id"].isna(), "kategori"].value_counts().to_dict())

Baris bersih: 90,818 | input split: 9
Judul tanpa id per kategori: {'inet': 5577}


## 2. Menangani id kosong pada kategori inet

Ada tiga pilihan. Membuang 5.577 judul inet membuat satu kategori hilang dari indeks, padahal judulnya utuh dan berguna untuk pencarian kata. Memakai nomor baris sebagai id juga rapuh, karena nomor itu berubah setiap kali data diurutkan atau ditambah. Saya memilih id pengganti yang stabil: `inet-` diikuti CRC32 dari judul dalam heksadesimal. CRC32 memberi nilai yang sama di setiap komputer, dan awalan `inet-` menandai bahwa id ini bukan nomor artikel detik sehingga tidak boleh dipakai untuk menyusun URL. Sel berikut memeriksa bahwa id pengganti itu tidak bertabrakan.

In [4]:
def id_dokumen(id_asli, judul, kategori):
    """id detik bila ada; bila kosong, id pengganti stabil dari CRC32 judul."""
    if pd.notna(id_asli):
        return str(int(id_asli))
    return f"{kategori}-{zlib.crc32(judul.encode('utf-8')):08x}"

inet = bersih[bersih["id"].isna()]
pengganti = [id_dokumen(i, j, k) for i, j, k in zip(inet["id"], inet["judul"], inet["kategori"])]
print("Judul tanpa id      :", f"{len(pengganti):,}")
print("id pengganti unik   :", f"{len(set(pengganti)):,}")
print("Tabrakan            :", len(pengganti) - len(set(pengganti)))
print("Contoh              :", pengganti[:3])

Judul tanpa id      : 5,577
id pengganti unik   : 5,577
Tabrakan            : 0
Contoh              : ['inet-461b08b9', 'inet-681767da', 'inet-e02109eb']


## 3. Mapper, shuffle, dan reducer

Mapper membaca satu split (satu kategori) dan menghasilkan (kata, id). Kata yang muncul dua kali dalam satu judul hanya dikirim sekali, karena indeks cukup mencatat bahwa judul itu memuat kata tersebut. Shuffle mengurutkan pasangan menurut kata, seperti `sort` pada Hadoop Streaming. Reducer menggabungkan id per kata menjadi daftar yang terurut: id numerik detik lebih dulu, lalu id pengganti.

In [5]:
import time

def tokenisasi(teks):
    return re.findall(r"[a-z]+", teks.lower())

def mapper_index(path):
    """Satu mapper = satu partisi. Keluaran: pasangan (kata, id)."""
    kategori = path.parent.name.split("=")[1]
    df_ = pd.read_parquet(path, columns=["judul", "id"])
    hasil = []
    for judul, id_asli in zip(df_["judul"], df_["id"]):
        doc = id_dokumen(id_asli, judul, kategori)
        for kata in sorted(set(tokenisasi(judul))):
            hasil.append((kata, doc))
    return hasil

def urutan_id(doc):
    return (0, int(doc), "") if doc.isdigit() else (1, 0, doc)

def reducer_index(kata, daftar_id):
    return kata, sorted(set(daftar_id), key=urutan_id)

t0 = time.perf_counter()
hasil_map = [mapper_index(p) for p in splits]                       # MAP
pasangan = sorted(p for h in hasil_map for p in h)                  # SHUFFLE (urut per kata)
indeks = {}
kunci, kumpul = None, []
for kata, doc in pasangan:                                          # REDUCE
    if kata != kunci:
        if kunci is not None:
            k, ids = reducer_index(kunci, kumpul); indeks[k] = ids
        kunci, kumpul = kata, []
    kumpul.append(doc)
if kunci is not None:
    k, ids = reducer_index(kunci, kumpul); indeks[k] = ids
print(f"Waktu MapReduce     : {time.perf_counter() - t0:.2f} detik")
print("Pasangan dari mapper:", f"{len(pasangan):,}")
print("Kata di indeks      :", f"{len(indeks):,}")

Waktu MapReduce     : 8.37 detik
Pasangan dari mapper: 817,993
Kata di indeks      : 32,275


## 4. Pemeriksaan

Indeks dicek silang dengan cara biasa: untuk tiap kata, jumlah id di indeks harus sama dengan jumlah judul yang memuat kata itu.

In [6]:
cek = collections.Counter()
for judul in bersih["judul"]:
    cek.update(set(tokenisasi(judul)))
print("Jumlah kata sama            :", len(cek) == len(indeks))
print("Panjang daftar id sama      :", all(len(indeks[k]) == n for k, n in cek.items()))

ada_pengganti = sum(any(not d.isdigit() for d in ids) for ids in indeks.values())
print("Kata yang memuat id inet-   :", f"{ada_pengganti:,}")

Jumlah kata sama            : True
Panjang daftar id sama      : True
Kata yang memuat id inet-   : 6,747


## 5. Tiga kata contoh beserta lima id pertama

In [7]:
for kata in ["corona", "banjir", "ponsel"]:
    ids = indeks[kata]
    print(f"{kata:<8} {len(ids):>6,} judul | 5 id pertama: {ids[:5]}")

corona   17,228 judul | 5 id pertama: ['4856495', '4856598', '4868317', '4868545', '4869163']
banjir    1,992 judul | 5 id pertama: ['4842198', '4842285', '4842286', '4842287', '4842292']
ponsel      343 judul | 5 id pertama: ['4855118', '4856610', '4858802', '4859790', '4865851']


Id numerik bisa dibuka kembali sebagai artikel. Sel berikut menampilkan judul untuk lima id pertama kata "banjir", dan contoh id pengganti untuk kata "ponsel" yang berasal dari kategori inet.

In [8]:
judul_per_id = {id_dokumen(i, j, k): j for i, j, k in zip(bersih["id"], bersih["judul"], bersih["kategori"])}
for doc in indeks["banjir"][:5]:
    print(f"{doc:<10} {judul_per_id[doc]}")
print()
for doc in [d for d in indeks["ponsel"] if not d.isdigit()][:3]:
    print(f"{doc:<14} {judul_per_id[doc]}")

4842198    Malam Tahun Baru 2020, 6 Desa di Bengkulu Terendam Banjir
4842285    Banjir Genangi Jl DI Panjaitan Jatinegara dan Jl Taman Mini Jaktim
4842286    Permukiman Tambun Selatan Kabupaten Bekasi Banjir, Warga Mengungsi
4842287    Tinggi Banjir Terus Naik, Warga Cipinang Pulomaja Jaktim Mengungsi
4842292    Video Banjir Genangi Permukiman di Jaktim dan Jakbar

inet-005bfd65  Galaxy A71, Ponsel Trendi Performa Mumpuni
inet-00c757d7  Sudah Diblokir, Kok Masih Banyak Ponsel BM di Pasaran?
inet-00db5396  Duh, Ponsel Samsung dan Huawei KW Banjiri AliExpress


## 6. Menyimpan ke siap_pakai

Indeks disimpan sebagai Parquet dengan kolom `kata`, `jumlah_judul`, dan `daftar_id`.

In [9]:
SIAP = Path("siap_pakai"); SIAP.mkdir(exist_ok=True)
indeks_df = pd.DataFrame({"kata": list(indeks),
                          "jumlah_judul": [len(v) for v in indeks.values()],
                          "daftar_id": list(indeks.values())})
indeks_df.to_parquet(SIAP / "inverted_index.parquet", index=False)
print(indeks_df.sort_values("jumlah_judul", ascending=False).head(5)[["kata", "jumlah_judul"]].to_string(index=False))
print("Ukuran berkas:", round((SIAP / "inverted_index.parquet").stat().st_size / 1e6, 1), "MB")

  kata  jumlah_judul
    di         26180
corona         17228
   ini         10085
  yang          8630
   dan          6722
Ukuran berkas: 2.6 MB


In [10]:
import shutil
from pathlib import Path

# Pastikan variabel DIR_SIMPAN sudah didefinisikan sebelumnya, contoh:
# DIR_SIMPAN = "/content/drive/MyDrive/BigData_P05"
SIMPAN = Path(DIR_SIMPAN)

# 1) Salin inverted index ke siap_pakai/ di Drive.
sumber_parquet = Path("siap_pakai/inverted_index.parquet")

if sumber_parquet.exists():
    target_dir = SIMPAN / "siap_pakai"
    target_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy(sumber_parquet, target_dir / "inverted_index.parquet")
    print("disalin  : siap_pakai/inverted_index.parquet")
else:
    print("dilewati : siap_pakai/inverted_index.parquet (berkas lokal tidak ditemukan)")


# 2) Buat ulang zip supaya inverted index ikut di dalamnya
def buat_zip(simpan, nama="BD_P05_2411537001_lapisan_data"):
    sementara = Path("/content/zip_sementara")
    shutil.rmtree(sementara, ignore_errors=True)
    sementara.mkdir(parents=True, exist_ok=True)

    # Salin folder terkurasi dan siap_pakai jika ada
    for folder in ["lapisan_terkurasi", "siap_pakai"]:
        if (simpan / folder).exists():
            shutil.copytree(simpan / folder, sementara / folder)

    # Salin 1 sampel berkas .jsonl dari lapisan_mentah
    folder_mentah_target = sementara / "lapisan_mentah"
    folder_mentah_target.mkdir(parents=True, exist_ok=True)

    folder_mentah_sumber = simpan / "lapisan_mentah"
    if folder_mentah_sumber.exists():
        berkas_jsonl = sorted(folder_mentah_sumber.glob("*.jsonl"))
        if berkas_jsonl:
            shutil.copy(berkas_jsonl[0], folder_mentah_target / berkas_jsonl[0].name)

    arsip = shutil.make_archive(str(simpan / nama), "zip", sementara)
    shutil.rmtree(sementara)
    return Path(arsip)


if (SIMPAN / "lapisan_terkurasi").exists():
    arsip = buat_zip(SIMPAN)
    print(f"Zip      : {arsip.name} ({arsip.stat().st_size / 1e6:.1f} MB)")
else:
    print(
        "Zip belum dibuat: jalankan notebook praktikum sampai sel terakhir lebih dulu."
    )

# 3) Isi siap_pakai/ di Drive
print("\nIsi siap_pakai/ di Drive:")
folder_siap_pakai = SIMPAN / "siap_pakai"
if folder_siap_pakai.exists():
    for f in sorted(folder_siap_pakai.iterdir()):
        print(f"  {f.name:<28} {f.stat().st_size / 1e6:5.1f} MB")
else:
    print("  (folder siap_pakai/ belum ada di Drive)")

disalin  : siap_pakai/inverted_index.parquet
Zip belum dibuat: jalankan notebook praktikum sampai sel terakhir lebih dulu.

Isi siap_pakai/ di Drive:
  inverted_index.parquet         2.6 MB
